### Creating a parser that takes in the params from the injection and uses it in the static_params section


In [1]:
from pycbc.workflow import WorkflowConfigParser
from pycbc.inference.models.gaussian_noise import GaussianNoise
from pycbc.inference.models.marginalized_gaussian_noise import MarginalizedHMPhase
from pycbc.inference.models import read_from_config
import h5py

/Users/vikasjadhav/anaconda3/envs/hmnumerical/lib/python3.11/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


Create a function that returns a configparser to load the model.

In [102]:

def create_parser(
        injection_file,variable_params_file,data_file,model_file,
        mode_array,**kwargs):
    """ 
    Create a Workflowconfigparser that updates static_params by reading it from 
    the injection file. 
    Inputs : 
    "injection file" : an hdf file containing only one injection
    "variable_params_file" : a config file containing section of variable_params
    and must include the priors of each parameter.
    "data_file" : a config file containing common data settings.The trigger-time and 
    injection-file options will be updated using the provided injection.
    "model_file" : config file with the model settings.
    "mode_array" : mode array to use in the signal model.
    "kwargs" : additional options for the [model] section.
    Output : 
    "cp" : a workflow config parser
    """
    cp = WorkflowConfigParser([data_file,variable_params_file,model_file])
    
    
    ## Read the injection file to gather all the params
    inj = h5py.File(injection_file,'r')
    all_params = {}
    for p in inj.keys():
        all_params[p] = inj[p][0]
    for sp in inj.attrs['static_args']:
        all_params[sp] = inj.attrs[sp]
    static_params = all_params.copy()
    ## Remove the params included in the variable params
    for vp in cp['variable_params'].keys():
        _ = static_params.pop(vp)
    
    ## Add the mode array to be used in the model
    static_params['mode_array'] = mode_array
    ## Create and add options to static_params section
    cp.add_section('static_params')
    for sp in static_params:
        cp.add_options_to_section('static_params',
                                  [(sp,str(static_params[sp]))])
    
    ## Update options to data section
    cp.add_options_to_section('data',[('trigger-time',str(static_params['tc'])),
                                      ('injection-file',injection_file)])
    for key, value in kwargs.items():
        cp.add_options_to_section('model', [(key, str(value))])
    return cp




Test to compare the existing model

In [121]:
marg_cp = create_parser(injection_file='injections/test_inj_v1.hdf',
                   variable_params_file='var_par.ini',
                   data_file='data_config/data_zeronoise.ini',
                   model_file='model_config/marg_model.ini',
                   mode_array='22')

hm_cp = create_parser(injection_file='injections/test_inj_v1.hdf',
                   variable_params_file='var_par.ini',
                   data_file='data_config/data_zeronoise.ini',
                   model_file='model_config/hm_model.ini',
                   mode_array='22')
marg_noise_cp = create_parser(injection_file='injections/test_inj_v1.hdf',
                   variable_params_file='var_par.ini',
                   data_file='data_config/data_noise.ini',
                   model_file='model_config/marg_model.ini',
                   mode_array='22')

hm_noise_cp = create_parser(injection_file='injections/test_inj_v1.hdf',
                   variable_params_file='var_par.ini',
                   data_file='data_config/data_noise.ini',
                   model_file='model_config/hm_model.ini',
                   mode_array='22')

In [122]:

import time

# Time marg_model
start = time.time()
marg_model = read_from_config(marg_cp)
marg_time = time.time() - start
print(f"marg_model loaded in {marg_time:.4f} seconds")

# Time hm_model
start = time.time()
hm_model = read_from_config(hm_cp)
hm_time = time.time() - start
print(f"hm_model loaded in {hm_time:.4f} seconds")

# Time marg_noise_model
start = time.time()
marg_noise_model = read_from_config(marg_noise_cp)
marg_noise_time = time.time() - start
print(f"marg_noise_model loaded in {marg_noise_time:.4f} seconds")

# Time hm_noise_model
start = time.time()
hm_noise_model = read_from_config(hm_noise_cp)
hm_noise_time = time.time() - start
print(f"hm_noise_model loaded in {hm_noise_time:.4f} seconds")

# Print summary
print(f"\nTotal loading time: {marg_time + hm_time + marg_noise_time + hm_noise_time:.4f} seconds")

marg_model loaded in 4.1688 seconds


hm_model loaded in 4.1083 seconds


marg_noise_model loaded in 10.6594 seconds
hm_noise_model loaded in 10.7259 seconds

Total loading time: 29.6624 seconds


In [123]:
marg_model.update(coa_phase = 0)
hm_model.update(coa_phase = 0)
marg_noise_model.update(coa_phase = 0)
hm_noise_model.update(coa_phase = 0)

In [124]:
import time

# Time marg_model.loglr
start = time.time()
marg_loglr = marg_model.loglr
marg_loglr_time = time.time() - start
print(f"marg_model.loglr evaluated in {marg_loglr_time:.4f} seconds: {marg_loglr}")

# Time hm_model.loglr
start = time.time()
hm_loglr = hm_model.loglr
hm_loglr_time = time.time() - start
print(f"hm_model.loglr evaluated in {hm_loglr_time:.4f} seconds: {hm_loglr}")

# Time marg_noise_model.loglr
start = time.time()
marg_noise_loglr = marg_noise_model.loglr
marg_noise_loglr_time = time.time() - start
print(f"marg_noise_model.loglr evaluated in {marg_noise_loglr_time:.4f} seconds: {marg_noise_loglr}")

# Time hm_noise_model.loglr
start = time.time()
hm_noise_loglr = hm_noise_model.loglr
hm_noise_loglr_time = time.time() - start
print(f"hm_noise_model.loglr evaluated in {hm_noise_loglr_time:.4f} seconds: {hm_noise_loglr}")

# Print summary
total_time = marg_loglr_time + hm_loglr_time + marg_noise_loglr_time + hm_noise_loglr_time
print(f"\nTotal loglr evaluation time: {total_time:.4f} seconds")

marg_model.loglr evaluated in 0.0251 seconds: 1575.9446272388795
using analytic approximation
0.0003630837891250849
hm_model.loglr evaluated in 0.0170 seconds: 1575.944503271679
marg_noise_model.loglr evaluated in 0.0161 seconds: 1645.5001434053906
using analytic approximation
0.0002924168948084116
hm_noise_model.loglr evaluated in 0.0133 seconds: 1645.5000663171895

Total loglr evaluation time: 0.0715 seconds
